# SilentOS — First genuine ISL sign-model training (Colab)

**Purpose:** run the *actual* SilentOS training pipeline on the *actual* 128-clip ISL
dataset and produce the first honestly trained sign-recognition model.

**Hard rules for this notebook (do not edit around them):**

1. Every number printed below is **computed by the cells at run time**. Nothing is
   pre-filled. If a cell does not run, there is no result — not a placeholder.
2. The MediaPipe gate (exactly 1 → 2 → 4 **cumulative** extractions) **must pass
   before training**. If it fails, the run **stops** — copy the failure evidence
   and follow the contingency note. Do not skip the gate, and do not train without
   it.
3. Training reuses `silentos.sign.train.train_sign` — the same function the backend
   Training Center calls. There is **no separate notebook feature pipeline**.
4. Splits are the repository's participant-held-out splits (seed 42). They are
   **verified, never regenerated** here.
5. Vocabulary is the 10 SilentOS words. Only classes present in the dataset with
   adequate data train: **7 of 10** (PAIN, DOCTOR, TOILET are absent from this
   dataset; STOP has only 4 clips and 0 test support — both reported honestly).

**Success criteria:** a `models/sign/` artifact set (`model.npz`, `model.onnx`,
`manifest.json`) produced by `train_sign`, verified by the production
`SignRecognizer` loader, with evaluation numbers copied verbatim from the returned
manifest into Drive.

## 1. Get the repository into Colab

`data/` and `models/` are **git-ignored** — a fresh clone has no clips and no
MediaPipe task assets. The cells below fetch them (dataset from Hugging Face,
task file from Google's model storage).

Pick **one** option and run it:

- **Option A — GitHub clone:** `git clone https://github.com/varun-pahuja/cornerstone.git`.
  **⚠ A clone contains only *committed* files.** Uncommitted source changes in
  your working tree (config fixes, the GPU-delegate experiment, etc.) will **not**
  be present. Use Option B if you need the exact current working tree.
- **Option B — ZIP of the current repository (recommended):** packs your working
  tree **including uncommitted changes**. On your machine, from the repo root:

  ```bash
  zip -r silentos.zip . -x ".venv/*" "node_modules/*" "data/external/*"       "data/raw/*" "*.mp4" "__pycache__/*" ".git/*" "frontend/dist/*"
  ```

  (`.git/*` is excluded on purpose — the ZIP *is* the uncommitted state; exclude
  `data/raw`/`*.mp4` only if you prefer Colab to re-download the 42 MB dataset
  via the cell in section 4. Keep them if you want zero re-download.)
- **Option C:** copy the repo into Google Drive first, then mount and `cd`.

The notebook records which source you used in `environment.json`.

In [ ]:
# Option A — GitHub clone (COMMITTED state only — uncommitted changes are MISSING)
!git clone https://github.com/varun-pahuja/cornerstone.git /content/cornerstone
%cd /content/cornerstone
# print which commit we got, so the record shows exactly what was cloned:
!git log -1 --oneline

# Option B — RECOMMENDED: upload a ZIP of the CURRENT working tree (includes
# uncommitted changes). Build the zip locally first (see previous cell), then:
# from google.colab import files
# import zipfile
# uploaded = files.upload()          # choose silentos.zip
# zipfile.ZipFile(next(iter(uploaded))).extractall("/content/silentos_zip")
# %cd /content/silentos_zip         # adjust if the zip root differs
# !ls                                # sanity-check: configs/ backend/ scripts/ present

# Option C — mount Drive and cd to your copy (uncomment):
# from google.colab import drive
# drive.mount("/content/drive")
# %cd "/content/drive/MyDrive/cornerstone"

## 2. Dependencies and environment compatibility

In [ ]:
%pip install -q "mediapipe>=0.10" opencv-python-headless "pyyaml>=6" "pydantic>=2" \
    numpy onnx onnxruntime psutil huggingface_hub

In [ ]:
import os, platform, shutil, subprocess, sys, time
from pathlib import Path

report: dict = {"python": sys.version.split()[0], "platform": platform.platform()}
assert sys.version_info >= (3, 10), f"Python >= 3.10 required, got {sys.version}"

import numpy, cv2, mediapipe, onnx, onnxruntime, pydantic, yaml, psutil
report.update({
    "numpy": numpy.__version__, "opencv": cv2.__version__,
    "mediapipe": mediapipe.__version__, "onnx": onnx.__version__,
    "onnxruntime": onnxruntime.__version__, "pydantic": pydantic.__version__,
})

vm = psutil.virtual_memory()
report["ram_total_gb"] = round(vm.total / 1e9, 1)
report["ram_available_gb"] = round(vm.available / 1e9, 1)
report["cpus"] = psutil.cpu_count(logical=False) or psutil.cpu_count()

gpu = shutil.which("nvidia-smi")
if gpu:
    out = subprocess.run([gpu, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
    report["gpu"] = out or "nvidia-smi present, no GPU listed"
else:
    report["gpu"] = "none (CPU runtime)"

print("ENVIRONMENT REPORT (real, measured at run time):")
for k, v in report.items():
    print(f"  {k}: {v}")

## 3. Locate the repository and load SilentOS settings

The notebook must run from the repository root so the relative paths in
`configs/default.yaml` (`./data`, `./models`) resolve exactly as they do on the
laptop.

In [ ]:
import os, sys
from pathlib import Path

def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents, Path("/content/cornerstone"), Path("/content")]:
        if (cand / "configs" / "default.yaml").is_file() and (cand / "backend" / "src").is_dir():
            return cand
    raise FileNotFoundError(
        "Repository root not found (needs configs/default.yaml + backend/src). "
        "Run the clone/upload cell first."
    )

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "backend" / "src"))

from silentos.config import load_settings
settings = load_settings(ROOT / "configs" / "default.yaml")
print("repo root:", ROOT)
print("vocabulary:", settings.vocabulary.labels)
print("abstain label:", settings.vocabulary.abstain_label)
assert len(settings.vocabulary.labels) == 10

## 4. Acquire the real dataset (idempotent)

Source: `vidit031/isl-isolated-40words` **pinned to revision**
`255ce7fd10da5d05bc30d59668ba256e5f02531c` (the revision verified in
`docs/isl40_coverage.md`) — 128 clips across the 7 SilentOS classes present
(research/education use; multi-source licenses documented in that coverage
report). Ingestion verifies every clip's SHA-256 against the upstream
`metadata.csv` and applies the participant-held-out splits.

In [ ]:
# Pinned dataset revision — must match docs/isl40_coverage.md exactly.
REVISION = "255ce7fd10da5d05bc30d59668ba256e5f02531c"
coverage_doc = ROOT / "docs" / "isl40_coverage.md"
assert coverage_doc.is_file(), f"missing {coverage_doc}"
doc_text = coverage_doc.read_text()
assert REVISION in doc_text, (
    f"REVISION {REVISION} does not appear in docs/isl40_coverage.md — "
    "the notebook and the coverage report disagree; fix before downloading.")
print(f"revision {REVISION} verified against docs/isl40_coverage.md")
print("dataset: vidit031/isl-isolated-40words @", REVISION)

In [ ]:
import subprocess, sys
from pathlib import Path

have_manifest = (ROOT / "data" / "manifests" / "isl40.json").is_file()
have_clips = any((ROOT / "data" / "raw").rglob("*.mp4"))
if have_manifest and have_clips:
    print("dataset already present — skipping download/ingest")
else:
    print(f"fetching target-class clips from Hugging Face @ {REVISION} ...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "vidit031/isl-isolated-40words",
        repo_type="dataset",
        revision=REVISION,          # pinned — never "main"
        local_dir=str(ROOT / "data" / "external" / "isl-isolated-40words"),
        allow_patterns=[
            "README.md", "metadata.csv",
            "help/*", "water/*", "food/*", "yes/*", "no/*", "stop/*", "thank_you/*",
        ],
    )
    print("ingesting (verifies sha256, registers, splits) ...")
    subprocess.run([sys.executable, "scripts/ingest_isl.py"], cwd=ROOT, check=True)

## 5. Verify manifests, splits, and SHA-256 checksums

These cells must all pass before anything is trained. They re-hash **every clip on
disk** against the manifest and re-check the participant-held-out split invariants.

In [ ]:
from collections import Counter, defaultdict
from silentos.dataset import load_manifest, manifest_path

m = load_manifest(manifest_path(settings.paths.manifests_dir, "isl40"))
EXPECTED_LABELS = {"HELP", "WATER", "FOOD", "YES", "NO", "STOP", "THANK YOU"}
EXPECTED_TOTAL = 128

labels = {c.label for c in m.clips}
assert m.clip_count == EXPECTED_TOTAL == len(m.clips), (
    f"expected {EXPECTED_TOTAL} clips, manifest has {len(m.clips)}")
assert labels == EXPECTED_LABELS, f"unexpected labels: {labels ^ EXPECTED_LABELS}"
assert all(c.modality == "sign" for c in m.clips)
assert m.splits, "manifest has no splits — run ingestion first"
assert m.splits["seed"] == 42, f"split seed changed: {m.splits['seed']}"

parts = m.splits["participants"]
tr = {p for p, s in parts.items() if s == "train"}
va = {p for p, s in parts.items() if s == "val"}
te = {p for p, s in parts.items() if s == "test"}
assert not (tr & va) and not (tr & te) and not (va & te), "participant leakage across splits!"
pinned = m.splits.get("pin", {})
assert all(parts.get(p) == "train" for p in pinned), "pinned source left train split"

by = defaultdict(Counter)
for c in m.clips:
    by[c.split][c.label] += 1
print(f"clips={len(m.clips)} participants: train={len(tr)} val={len(va)} test={len(te)}")
print(f"pinned train-only sources: {sorted(pinned)}")
for s in ("train", "val", "test"):
    print(f"  {s:5s} {dict(sorted(by[s].items()))}")
stop_n = sum(by[s]["STOP"] for s in by)
print(f"NOTE: STOP has {stop_n} clips total and "
      f"{by['test']['STOP']} in test — per-class STOP test metrics are not computable.")
print("manifest + split verification: PASS")

In [ ]:
import hashlib
from pathlib import Path

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

missing, mismatched, checked = [], [], 0
for c in m.clips:
    p = settings.paths.data_dir / c.file
    if not p.is_file():
        missing.append(c.file)
        continue
    if sha256_file(p) != c.sha256:
        mismatched.append(c.file)
    checked += 1
print(f"sha256 checked={checked} missing={len(missing)} mismatched={len(mismatched)}")
for f in (missing + mismatched)[:10]:
    print("  BAD:", f)
assert not missing and not mismatched, "dataset integrity failure — re-run ingestion"
print("SHA-256 verification: PASS (all 128 clips match the manifest)")

## 6. MediaPipe gate — exactly 1, then 2, then 4 **cumulative** extractions

This is the checkpoint that failed on the laptop (external SIGKILL at ~3 s).
One extractor instance is kept for the whole test; the three checkpoints are
**cumulative totals** (1, then 2, then 4 extractions — 4 in total, not 1+2+4=7).
Results are written to `mediapipe_test_result.json` **after every checkpoint**, so
a killed kernel still leaves evidence of exactly where it stopped.

- `hand=N` only means no hand was visible in that frame — the gate is about
  **the process surviving extraction**, not about detection.
- The gate **passes** only if the JSON ends with `"status": "PASSED"`,
  `"extractions_total": 4`, and checkpoints for 1, 2 and 4 are all present.

**How to distinguish outcomes (read `mediapipe_test_result.json`):**

| JSON state | Meaning | Next action |
|---|---|---|
| `status: PASSED`, `extractions_total: 4`, checkpoints 1/2/4 present | gate genuinely passed | continue to training |
| `status: FAILED` + `error` field | Python exception during extraction | STOP, keep evidence |
| `status: running` (no final status), `extractions_total` 1–3 | **runtime was killed externally** mid-gate (exactly the laptop SIGKILL signature) | STOP — this is failure evidence, not a pass; do not train |
| file missing entirely | kernel died before the first flush | STOP, keep the notebook output |

In [ ]:
import json, os, time, urllib.request
from pathlib import Path
import cv2

# Colab/headless hosts have no EGL: the GPU delegate aborts the process inside
# HandLandmarker.create_from_options() (a C++ LOG(FATAL) Python cannot catch).
# Pin the CPU delegate here; the env var is recorded in mediapipe_test_result.json.
os.environ["SILENTOS_MEDIAPIPE_DELEGATE"] = "cpu"

from silentos.sign.landmarks import HandExtractor, LandmarkUnavailable

GATE_PATH = ROOT / "mediapipe_test_result.json"
gate: dict = {"steps": {}, "status": "running", "extractions_total": 0,
        "delegate": os.environ.get("SILENTOS_MEDIAPIPE_DELEGATE", "default")}

def rss_mb() -> int:
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmRSS"):
                return int(line.split()[1]) // 1024
    return -1

def flush_gate() -> None:
    gate["rss_mb"] = rss_mb()
    GATE_PATH.write_text(json.dumps(gate, indent=2))

# Start from a clean evidence file so a stale PASSED from an earlier run cannot
# be mistaken for this run's result.
if GATE_PATH.is_file():
    GATE_PATH.unlink()
flush_gate()

task = settings.paths.assets_dir / "hand_landmarker.task"
if not task.is_file():
    task.parent.mkdir(parents=True, exist_ok=True)
    URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
           "hand_landmarker/float16/1/hand_landmarker.task")
    print("downloading hand_landmarker.task ...")
    urllib.request.urlretrieve(URL, task)
gate["task_file"] = str(task)
flush_gate()

# pick a real clip and read its first 4 frames
clips = sorted(settings.paths.data_dir.rglob("*.mp4"))
assert clips, "no clips found — run the dataset cell first"
cap = cv2.VideoCapture(str(clips[0]))
frames = []
while len(frames) < 4:
    ok, fr = cap.read()
    if not ok:
        break
    frames.append(fr)
cap.release()
assert len(frames) >= 4, f"could only read {len(frames)} frames from {clips[0].name}"
gate["clip"] = clips[0].name
flush_gate()

try:
    t_all = time.time()
    ex = HandExtractor(task)          # ONE instance for the whole cumulative test
    for target in (1, 2, 4):          # cumulative totals, not per-step counts
        t0 = time.time()
        hands = 0
        while gate["extractions_total"] < target:
            fr = frames[gate["extractions_total"]]
            lm = ex.extract(fr)
            gate["extractions_total"] += 1
            hands += int(lm is not None)
        gate["steps"][str(target)] = {
            "ok": True,
            "cumulative_extractions": gate["extractions_total"],
            "seconds_in_step": round(time.time() - t0, 3),
            "hands_detected_in_step": hands,
        }
        flush_gate()          # written BEFORE the next step — survives a kill
        print(f"cumulative {target} extractions: OK "
              f"(step {gate['steps'][str(target)]['seconds_in_step']}s, "
              f"hands={hands}, total={gate['extractions_total']}, rss={rss_mb()}MB)")
    assert gate["extractions_total"] == 4, gate["extractions_total"]
    assert set(gate["steps"]) == {"1", "2", "4"}, gate["steps"]
    gate["status"] = "PASSED"
    gate["total_seconds"] = round(time.time() - t_all, 3)
    flush_gate()
    print(f"MEDIAPIPE GATE: PASSED — {gate['extractions_total']} total extractions "
          "over checkpoints 1, 2, 4 — safe to continue to training.")
except Exception as e:
    gate["status"] = "FAILED"
    gate["error"] = f"{type(e).__name__}: {e}"
    flush_gate()
    raise RuntimeError(
        "STOP: MediaPipe gate failed. Do NOT train. Save "
        f"{GATE_PATH} and the notebook output as failure evidence, then follow "
        "the contingency section (evaluate an alternative landmark backend)."
    ) from e

### If the gate stopped here

The kernel dying with **SIGKILL / Killed** at this step reproduces the laptop
blocker (see `docs/evidence/t-ml-01-2026-10-30/FINDINGS.md`). What to do:

1. Keep `mediapipe_test_result.json` + this notebook's output — that is the
   evidence that MediaPipe is not viable in this runtime either.
   - If the file says `"status": "running"` (or the kernel died before any final
     status was written) the runtime was **killed externally** mid-gate; the
     checkpoints present show exactly how many extractions survived. That is a
     gate **failure**, not a pass. A `"status": "FAILED"` file means a Python
     exception was caught and recorded instead.
   - Only `"status": "PASSED"` with `"extractions_total": 4` means the gate
     succeeded; anything else is a stop.
2. **Stop this notebook.** Contingency (project task T-ML-10): evaluate an
   alternative hand-landmark backend — ONNX-compatible keypoint/hand models
   (e.g. MediaPipe-compatible or research hand-pose models), checking:
   Raspberry Pi 4 GB RAM fit, license/redistribution terms, **feature
   preprocessing parity** (`sign/features.py` sign_v1: wrist-relative,
   scale-normalized, 32-frame resample → 2016-dim), and measured latency/memory.
   No replacement ships until it passes a real end-to-end test through
   `train_sign` → `SignRecognizer`.
3. If the gate **passed**: continue — the local blocker does not apply to this
   runtime.

## 7. Train with the real SilentOS pipeline

`train_sign` = decode clips → MediaPipe landmarks → `sequence_to_features`
(`sign_v1`, 2016-dim) → seed-deterministic `LogisticBaseline` → honest per-split
evaluation → export `model.npz` + `model.onnx` + `manifest.json`.

Checkpoints: the baseline is full-batch gradient descent (no intermediate
weights), so the per-epoch loss log below **is** the training record; the final
artifact set is the checkpoint. Nothing here fabricates progress.

In [ ]:
from silentos.sign.train import train_sign, TrainingError
import json, time

EPOCHS_LOG: list[str] = []

def progress(epoch: int, total: int, loss: float) -> None:
    line = f"epoch {epoch}/{total} loss={loss:.6f}"
    EPOCHS_LOG.append(line)
    print(line, flush=True)

t0 = time.time()
try:
    manifest_out = train_sign(
        settings=settings,
        dataset_id="isl40",
        seed=42,
        epochs=40,
        lr=0.5,
        l2=1e-3,
        progress_cb=progress,
    )
except TrainingError as e:
    print(f"TRAINING FAILED (honest, no artifact written): {e}")
    raise
train_seconds = round(time.time() - t0, 1)
print(f"training finished in {train_seconds}s")
print(f"model version={manifest_out['version']} classes={manifest_out['labels']}")

(ROOT / "training_log.txt").write_text("\n".join(EPOCHS_LOG) + "\n")

## 8. Evaluation — exactly what the trainer computed

In [ ]:
import json

ev = manifest_out["evaluation"]
print("classes:", ev["classes"])
print("clips_used:", ev["clips_used"], "| clips_skipped:", ev["clips_skipped"])
print("final_loss:", ev["final_loss"])
for split in ("train", "val", "test"):
    block = ev.get(split)
    if block is None:
        print(f"{split}: honestly absent (no clips)")
        continue
    if "accuracy" not in block:
        print(f"{split}: {block}")
        continue
    print(f"{split}: accuracy={block['accuracy']} macro_f1={block['macro_f1']} "
          f"support={block['support']}")
    print(f"  confusion (rows=true): {block['confusion_matrix']}")
    for cls, row in block["per_class"].items():
        print(f"  {cls:9s} P={row['precision']} R={row['recall']} "
              f"F1={row['f1']} support={row['support']}")

# Persist the raw evaluation verbatim for the artifact bundle
(ROOT / "evaluation.json").write_text(json.dumps(ev, indent=2))
print("\nsaved evaluation.json (verbatim trainer output)")

## 9. Save artifacts to persistent storage (Google Drive)

Copies the real artifact set + logs + evaluation + environment + gate result to
`Drive/MyDrive/SilentOS_artifacts/<version>/`. Falls back to `/content` if Drive
is unavailable (then **download the folder manually** before the session ends).

In [ ]:
import json, shutil, time
from pathlib import Path

OUT = Path("/content/SilentOS_artifacts") / manifest_out["version"]
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/SilentOS_artifacts") / manifest_out["version"]
except Exception as e:
    print(f"Google Drive unavailable ({type(e).__name__}: {e}) — using {OUT}")
OUT.mkdir(parents=True, exist_ok=True)

sign_dir = settings.paths.models_dir / "sign"
written = []
for name in ("model.npz", "model.onnx", "manifest.json"):
    shutil.copy2(sign_dir / name, OUT / name)
    written.append(str(OUT / name))
for name in ("training_log.txt", "evaluation.json", "mediapipe_test_result.json"):
    src = ROOT / name
    if src.is_file():
        shutil.copy2(src, OUT / name)
        written.append(str(OUT / name))
(OUT / "environment.json").write_text(json.dumps({
    "environment_report": report,
    "train_seconds": train_seconds,
    "model_version": manifest_out["version"],
    "dataset": f"isl40 (vidit031/isl-isolated-40words @ {REVISION})",
    "trained_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "mediapipe_gate": json.loads(GATE_PATH.read_text()),
}, indent=2))
written.append(str(OUT / "environment.json"))
print("written:")
for w in written:
    print(" ", w)

## 10. Verify the export with the production loader

Uses `SignRecognizer` — the exact class the live recognition loop instantiates —
plus an `onnxruntime` smoke check of the exported ONNX graph, then runs real
frames from a **held-out test clip** through `predict_frame` and records whatever
comes out (including abstentions; nothing is forced).

In [ ]:
import json, os, time
from pathlib import Path
import numpy as np, cv2

# CPU delegate again (see the gate cell): a restarted kernel must not fall back
# to the GPU delegate, which aborts on Colab's missing EGL.
os.environ.setdefault("SILENTOS_MEDIAPIPE_DELEGATE", "cpu")

from silentos.sign.pipeline import SignRecognizer

model_path = settings.paths.models_dir / "sign" / "model.npz"
rec = SignRecognizer(model_path)
assert rec.available, "production loader could not load model.npz"
assert rec.version == manifest_out["version"], (
    f"version mismatch: loader={rec.version} trainer={manifest_out['version']}")
vocab = set(settings.vocabulary.labels)
assert set(rec.labels) <= vocab, f"loader labels outside vocabulary: {set(rec.labels) - vocab}"
print(f"SignRecognizer OK: version={rec.version} labels={rec.labels}")

# ONNX smoke check (edge artifact): one forward pass, probs sum to 1
import onnxruntime as ort
sess = ort.InferenceSession(str(settings.paths.models_dir / "sign" / "model.onnx"))
X = np.zeros((1, 2016), dtype=np.float32)
inp = sess.get_inputs()[0].name
probs = sess.run(None, {inp: X})[0]
assert probs.shape[-1] == len(rec.labels), f"onnx output shape {probs.shape}"
print(f"onnx OK: output shape={probs.shape} probs_sum={float(np.sum(probs)):.4f}")

# real inference on a held-out test clip
test_clips = [c for c in m.clips if c.split == "test"]
assert test_clips, "no test clips in manifest"
c = test_clips[0]
clip_path = settings.paths.data_dir / c.file
print(f"held-out clip: {c.file} (label={c.label}, participant={c.participant_id})")

cap = cv2.VideoCapture(str(clip_path))
rows, i = [], 0
t0 = time.time()
while True:
    ok, fr = cap.read()
    if not ok:
        break
    label, conf, _ = rec.predict_frame(fr)
    if label is not None:
        rows.append({"frame": i, "label": label, "confidence": round(conf, 4)})
    i += 1
cap.release()
rec.close()
print(f"frames_read={i} predictions={len(rows)} elapsed_s={round(time.time()-t0, 2)}")
for r in rows[:10]:
    print(" ", r)
if not rows:
    print("no predictions (all abstained / insufficient detections) — recorded honestly")

(ROOT / "loader_verification.json").write_text(json.dumps({
    "model_version": rec.version if hasattr(rec, "version") else manifest_out["version"],
    "loader_labels": rec.labels,
    "clip": c.file,
    "clip_true_label": c.label,
    "frames_read": i,
    "predictions": rows,
}, indent=2))
print("saved loader_verification.json")

## 11. Next steps (after a real, successful run)

1. Copy `SilentOS_artifacts/<version>/` back to the laptop's `models/sign/`
   (the production loader expects `model.npz` + `manifest.json` side by side).
2. Project tasks unlocked: **T-ML-06** (held-out evaluation on the laptop),
   **T-ML-07** (live recognition E2E), **T-DOC-03** (training-run report).
3. Still missing for the full 10-word vocabulary: **PAIN, DOCTOR, TOILET**
   (T-DATA-03, consented local ISL recording) — this notebook will not paper
   over them. Lip reading is a **separate** task with its own dataset (T-DATA-04).
4. Nothing in this notebook changes the dashboard or existing functionality.

### Honesty footer

All metrics shown above were computed by `train_sign` from real clips at run
time. If any cell did not run, there is no result for it. Synthetic smoke tests
in the backend suite are not evidence of accuracy — this run is.